# KV Cache: build it, prove it, watch the speedup grow

Imagine writing an essay where, before you add each new word, you re-read the entire essay from the
very first word just to decide what comes next. Write word 500 and you've re-read 499 words; write
word 1,000 and you've re-read 999. That is *exactly* what a transformer does when it generates text
**without a KV cache**: every new token reruns attention over every token before it, recomputing the
same numbers it already computed a step ago.

The **KV cache** is the fix — it stores each token's key and value vectors the first time they're
computed and reuses them forever, so each step does only the work for the *one* new token. It's a
**speed-and-memory mechanism, not a modeling one**: with or without it, the model produces the
*identical* tokens; only the speed and the VRAM change.

This notebook builds the cache from scratch in a few small pieces. **By the end you'll have built the
cache from scratch, watched K and V accumulate one row per token, proven the output is identical with
and without it, and measured the speedup growing as the sequence gets longer.**

## The problem: autoregressive decoding repeats itself

LLMs generate **autoregressively** — one token at a time, each new token conditioned on every token
before it. To produce token $t$, the model forms a **query** for the current position and compares it
against a **key** for every previous position, then mixes the corresponding **values**.

The catch is the *next* step. Naively, to produce token $t+1$ you run the whole sequence through the
model again — recomputing the keys and values for tokens $1 \dots t$, *the very same K and V you
computed one step ago*.

> In a causal (decoder-only) transformer, a token's key and value depend **only on that token and the
> ones before it** — never on anything after. So once token 5's K and V are computed, they are
> **frozen for the rest of the generation**. Recomputing them is pure waste.

How much waste? At decode step $t$ the naive approach re-projects K and V for all $t$ positions; summed
over a sequence of length $n$ that's $1 + 2 + \dots + n = \tfrac{n(n+1)}{2}$ projections —
**quadratic** in sequence length. The cache turns each step into a single new projection — **linear**.

## What it is, and why K and V but never Q

A **KV cache** is a per-layer buffer storing the **key** and **value** vectors of every token processed
so far. Generation splits into two phases:

- **Prefill** — process the entire prompt in one parallel pass, computing and storing K and V for all
  prompt tokens. This fills the cache.
- **Decode** — generate one token at a time. Each step computes K and V for **only the single new
  token**, appends them to the cache, and attends over the whole cache.

**Why K and V, not Q?** When the model generates a new token, that token's *query* asks a question of
*all the past tokens' keys* and gathers *all the past tokens' values*. But the past tokens' queries have
already done their job and will never be asked again. So you keep what future tokens need (K and V) and
discard what's spent (past Q). The clean one-liner: **Q is per-step and disposable; K and V are
per-token and reused.**

## How it works: the cache tensor and the append

The cache for one layer is a pair of tensors shaped `[batch, n_kv_heads, seq_len, head_dim]` — one for
K, one for V — held **per layer**. Lifecycle:

1. **Prefill.** Run the prompt (length $N$) through every layer once; each layer projects K and V for
   all $N$ tokens and writes them into its cache.
2. **Decode step.** Take the single newest token; project its $q, k, v$; **append** $k, v$ to the
   cache (now length $N{+}1$); compute attention of the one query against the full cached K/V.
3. **Repeat** until end-of-sequence.

In a real engine "append" is an **in-place write into a pre-allocated buffer**, not a fresh allocation.
Growing the tensor with a `torch.cat` *every step* — as the teaching code below does for clarity — would
re-allocate and copy the whole cache each token, turning the $O(n)$ win back into an $O(n^2)$ disaster.
The note in `decode_with_cache` flags exactly this.

> **Step 1 — set up one attention layer.** Everything below runs on a single self-attention layer, the smallest unit that shows the cache. We pick the device at runtime (CUDA → MPS → CPU so it runs anywhere), seed for reproducibility, and define the layer's size: `d_model=512` split into `8` heads of `64`. We also precompute `scale = 1/√head_dim`, the standard factor that keeps attention scores from blowing up as `head_dim` grows.
>
> *We do correctness on this device, but the timing sweep later deliberately drops to CPU — on a tiny single-layer model the GPU/MPS kernel-launch overhead swamps the math and flattens the speedup, so CPU is where the O(n²)->O(n) trend shows cleanest.*

In [1]:
import time
import torch
import torch.nn.functional as F

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("device:", device)
print("torch:", torch.__version__)

torch.manual_seed(0)
d_model, n_heads, head_dim = 512, 8, 64
scale = head_dim ** -0.5            # 1/sqrt(head_dim) — attention score scaling
assert n_heads * head_dim == d_model
Wq = torch.randn(d_model, d_model, device=device) * 0.02
Wk = torch.randn(d_model, d_model, device=device) * 0.02
Wv = torch.randn(d_model, d_model, device=device) * 0.02

device: mps
torch: 2.12.0


> **Step 2 — the head-split helper.** Attention runs in parallel across heads. A token is one `d_model`-long vector; `split_heads` reshapes a `(T, d_model)` block into `(n_heads, T, head_dim)` so each head sees its own `head_dim`-wide slice. This one reshape is used by every projection below — keep its output shape in mind: **heads first, then tokens, then per-head width.**

In [2]:
def split_heads(x):                 # (T, d_model) -> (n_heads, T, head_dim); transpose puts heads first so each head attends independently
    T = x.shape[0]
    return x.view(T, n_heads, head_dim).transpose(0, 1)

> **Step 3 — sanity-check the shape before trusting it.** Run the helper on 3 fake tokens and confirm we get `(8, 3, 64)`: 8 heads, 3 tokens, 64-wide each. Cheap shape checks like this catch the most common transformer bug — a silent wrong `view`/`transpose` — before it corrupts everything downstream.

In [3]:
demo = torch.randn(3, d_model, device=device)
print("split_heads(3 tokens):", tuple(split_heads(demo).shape))   # (8, 3, 64)

split_heads(3 tokens): (8, 3, 64)


> **Step 4 — one attention step.** This is the operation both decode loops share. A single query token (`q_t`) scores itself against every cached key, softmaxes those scores into weights, and mixes the cached values accordingly. Note what it takes: a query for **one** position, but `K` and `V` for **all** cached positions — that asymmetry is the whole reason we cache K and V and not Q.

In [4]:
# One attention step: a single query token attends over all cached keys/values.
def attn_step(q_t, K, V):           # q_t:(n_heads,1,head_dim)  K,V:(n_heads,T,head_dim)
    scores = (q_t @ K.transpose(-1, -2)) * scale   # q·kᵀ = one score per cached key; *scale stops large head_dim saturating softmax
    return (F.softmax(scores, dim=-1) @ V).transpose(0, 1).reshape(1, d_model)  # softmax->weights; @V mixes values; transpose+reshape rejoins heads into one d_model vector (transpose first so heads sit next to head_dim before the flatten)

> **Step 5a — the naive loop: recompute everything.** Here is decoding *without* a cache. At every step `t` it rebuilds K and V for **all** tokens `0..t` from scratch (`ctx @ Wk`, `ctx @ Wv`) — re-deriving vectors it already computed on every previous step. Summed over the sequence that redundant work is quadratic. Read the line marked `recomputed from scratch`: that is the waste the cache will delete.

In [5]:
def decode_no_cache(emb, N, w_q, w_k, w_v):   # re-projects K,V for ALL tokens each step -> O(n^2)
    outs = []
    for t in range(1, N + 1):
        ctx = emb[:t]
        K, V = split_heads(ctx @ w_k), split_heads(ctx @ w_v)   # recomputed from scratch
        outs.append(attn_step(split_heads(emb[t - 1:t] @ w_q), K, V))
    return torch.cat(outs, 0)

> **Step 5b — the cached loop: project only the new token.** Same output, different work. Each step projects K and V for **just the one new token**, appends them to a running cache, and attends over the whole cache. Compare it line-for-line with the loop above: the only change is that K and V come from `new` (one token) instead of `ctx` (all tokens). **Read the `NOTE`:** growing the cache with `torch.cat` every step is a teaching shortcut — it secretly reallocates the whole buffer, which would re-introduce the O(n²) cost. Real engines write in place into a pre-allocated buffer.

In [6]:
def decode_with_cache(emb, N, w_q, w_k, w_v):   # project only the NEW token, append -> O(n)
    outs, K_cache, V_cache = [], None, None
    for t in range(1, N + 1):
        new = emb[t - 1:t]
        k_new, v_new = split_heads(new @ w_k), split_heads(new @ w_v)
        # cat-per-step is the O(n^2) re-allocation trap; real engines write in place.
        K_cache = k_new if K_cache is None else torch.cat([K_cache, k_new], dim=1)
        V_cache = v_new if V_cache is None else torch.cat([V_cache, v_new], dim=1)
        outs.append(attn_step(split_heads(new @ w_q), K_cache, V_cache))
    return torch.cat(outs, 0)

> **Step 5c — a timing helper.** Before we measure anything, a tiny `timeit`: one warmup run (so we don't time cold caches/JIT), then the mean wall-clock over a few reps. We'll use it in the sweep further down.

In [7]:
def timeit(fn, reps=3):
    fn()  # warmup
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - t0) / reps * 1e3

## Watch the cache grow

The single most important thing to *see* is the cache growing by exactly one row per decode step.
Here's a tiny 3-token toy (one head, `head_dim=4`) that runs the decode loop step by step and prints
the cache shape after each append. Watch the `seq_len` dimension climb `1 → 2 → 3` — that's the cache
accumulating one token's K and V at a time. On the final step the single query attends over all 3
cached keys, so the attention-weights shape is `(1, 1, 3)`: **one query, over three keys**.

> **Step 6a — a 3-token toy, sized to read by eye.** To *see* the cache grow we shrink everything: 1 head, `head_dim=4`, 3 tokens. The shapes are now small enough to read directly. This cell only builds the toy weights and embeddings — no caching yet.
>
> *The loop in the next cell is line-for-line the inner loop of `decode_with_cache` from Step 5b — only the sizes change (1 head, head_dim=4, 3 tokens) so every shape is small enough to print and read.*

In [8]:
torch.manual_seed(0)
toy_dmodel, toy_heads, toy_hdim, toy_N = 4, 1, 4, 3
toy_scale = toy_hdim ** -0.5
toy_Wk = torch.randn(toy_dmodel, toy_dmodel, device=device) * 0.5
toy_Wv = torch.randn(toy_dmodel, toy_dmodel, device=device) * 0.5
toy_Wq = torch.randn(toy_dmodel, toy_dmodel, device=device) * 0.5
toy_emb = torch.randn(toy_N, toy_dmodel, device=device)


def toy_split(x):                   # (T, d_model) -> (n_heads=1, T, head_dim)
    return x.view(x.shape[0], toy_heads, toy_hdim).transpose(0, 1)

> **Step 6b — watch `seq_len` climb 1 → 2 → 3.** Now run the decode loop and print the cache shape after each append. The middle dimension is `seq_len`: it grows by exactly one row per step — that *is* the cache accumulating one token's K and V at a time. On the final step the single query attends over all 3 cached keys, so the attention-weights shape is `(1, 1, 3)`: **one query, three keys.**

In [9]:
K_cache = V_cache = None
for t in range(1, toy_N + 1):
    new = toy_emb[t - 1:t]                                   # the one new token this step
    k_new, v_new = toy_split(new @ toy_Wk), toy_split(new @ toy_Wv)
    K_cache = k_new if K_cache is None else torch.cat([K_cache, k_new], dim=1)
    V_cache = v_new if V_cache is None else torch.cat([V_cache, v_new], dim=1)
    print(f"step {t}: appended token {t-1} -> K_cache.shape = {tuple(K_cache.shape)}")

q = toy_split(toy_emb[toy_N - 1:toy_N] @ toy_Wq)            # (1, 1, 4)
attn_weights = F.softmax((q @ K_cache.transpose(-1, -2)) * toy_scale, dim=-1)
print(f"\nfinal query attends over {K_cache.shape[1]} cached keys "
      f"-> attn_weights.shape = {tuple(attn_weights.shape)}  (1 query, 3 keys)")

step 1: appended token 0 -> K_cache.shape = (1, 1, 4)
step 2: appended token 1 -> K_cache.shape = (1, 2, 4)
step 3: appended token 2 -> K_cache.shape = (1, 3, 4)



final query attends over 3 cached keys -> attn_weights.shape = (1, 1, 3)  (1 query, 3 keys)


## First, prove the cache changes nothing

Before timing anything, settle the only correctness question that matters: does caching change the
output? Run both decode paths on the small toy and assert they match to floating-point tolerance. If
this assert ever fails, the cache is doing something it shouldn't — it must be a pure speed trick.

We assert `atol=1e-5`. The two paths feed identical values into identical attention math; the only
difference is float rounding from projecting K/V batched (no-cache) versus one row at a time (cache) —
at most ~1e-7, and here just ~1e-8 (the printed max-diff). On some hardware it is even bitwise-identical.
`1e-5` sits comfortably above all of these without being so loose it would hide a real bug.

In [10]:
# Identical-output check on a small sequence, using the main (512-dim) layer from Steps 1-4.
emb_small = torch.randn(64, d_model, device=device) * 0.1
out_no = decode_no_cache(emb_small, 64, Wq, Wk, Wv)
out_yes = decode_with_cache(emb_small, 64, Wq, Wk, Wv)
assert torch.allclose(out_no, out_yes, atol=1e-5), "cache changed the output!"
print("identical outputs (no-cache vs cache):", torch.allclose(out_no, out_yes, atol=1e-5))
print("max abs difference:", (out_no - out_yes).abs().max().item())

identical outputs (no-cache vs cache): True
max abs difference: 1.7695128917694092e-08


## Now scale it up: watch the speedup grow

Correctness is settled, so now measure *speed* across growing sequence lengths. Read two things off
the table:

1. the **`identical` column is `True` at every length** — the cache still changes nothing about *what*
   the model produces; and
2. the **speedup widens** as the sequence grows — that widening *is* the $O(n^2) \to O(n)$ curve made
   visible. The no-cache loop's per-step work grows with position, so its total climbs faster than the
   cache's, and the ratio keeps opening up.

The sweep runs on **CPU** so the per-step arithmetic dominates and the widening trend is clean; on a
GPU/MPS accelerator the tiny single-layer tensors are dwarfed by kernel-launch overhead, which flattens
the ratio. Trust the *shape* — a ratio that grows with length — not any single row's exact milliseconds.

In [11]:
# Timing sweep on CPU (the trend is clearest where per-step math dominates).
# Time on CPU: per-step math dominates there, so the O(n^2)->O(n) trend is cleanest.
# (CPU copies are local — the Step-1 accelerator weights are left untouched.)
Wq_c, Wk_c, Wv_c = Wq.cpu(), Wk.cpu(), Wv.cpu()

print(f"{'N':>6} | {'no-cache':>10} | {'kv-cache':>10} | {'speedup':>8} | identical")
print("-" * 58)
for N in (256, 512, 1024, 2048):
    emb = torch.randn(N, d_model) * 0.1                      # CPU tensor
    a, b = decode_no_cache(emb, N, Wq_c, Wk_c, Wv_c), decode_with_cache(emb, N, Wq_c, Wk_c, Wv_c)
    same = torch.allclose(a, b, atol=1e-5)
    ms_no = timeit(lambda e=emb, n=N: decode_no_cache(e, n, Wq_c, Wk_c, Wv_c))
    ms_yes = timeit(lambda e=emb, n=N: decode_with_cache(e, n, Wq_c, Wk_c, Wv_c))
    print(f"{N:>6} | {ms_no:>8.1f}ms | {ms_yes:>8.1f}ms | {ms_no / ms_yes:>6.1f}x | {same}")

     N |   no-cache |   kv-cache |  speedup | identical
----------------------------------------------------------


   256 |     61.1ms |     49.3ms |    1.2x | True


   512 |    167.9ms |    116.2ms |    1.4x | True


  1024 |    486.3ms |    293.8ms |    1.7x | True


  2048 |   1691.0ms |    833.8ms |    2.0x | True


## Try it yourself

Before you change anything, **predict**: the demo uses `n_heads=8, head_dim=64`. If you bump
`n_heads` to 16 (keeping `head_dim=64`, so `d_model` doubles to 1024), does the **speedup curve** —
the *ratio* of no-cache to kv-cache time at each `N` — shift up, shift down, or stay roughly the same?

Then run it and check. *Hint:* both paths do more work per step, but the *redundant* work the cache
removes still grows like $O(n^2)$ in `N` either way — so the curve's **shape** is driven by length,
not head count. Heavier heads change the absolute milliseconds, not the reason the ratio widens.

> To see the real thing, call `model.generate(..., use_cache=True)` vs `use_cache=False` in Hugging
> Face on a long generation and watch the wall-clock diverge. Same idea, full model.

## What we saw

- **The cache grew by exactly one row per token** — `seq_len` climbed `1 → 2 → 3` in the toy, and the
  lone query attended over all cached keys (`attn_weights` shape `(1, 1, 3)`).
- **The output was identical** with and without the cache, at every length — the assert never fired.
  The cache is a *speed-and-memory* trick, not a modeling change.
- **The speedup widened** as the sequence grew, because the no-cache work is quadratic in length while
  the cache makes it linear — so the ratio between them keeps opening up.

**What we skipped** (and where to read it): the in-place pre-allocated buffer that real engines use
instead of `torch.cat` per step; the memory math that decides how many requests fit on a GPU; and the
production levers (GQA/MQA/MLA, PagedAttention, quantized caches, sliding-window). All of that is in the
[KV Cache concept page](../05-KV-Cache.md), with its [curated references](../05-KV-Cache.references.md).